In [8]:
!pip install -q vaderSentiment afinn nrclex transformers tqdm scipy

from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
from tqdm import tqdm
tqdm.pandas()

DATA_PATH = "/content/drive/MyDrive/312Final/newsroom_analysis_ready.csv"
OUTPUT_DIR = "/content/drive/MyDrive/312Final/"

df_full = pd.read_csv(DATA_PATH)
print(f"全量: {len(df_full)} 行")
print(df_full.columns.tolist())

TEXT_COL = "text"
PUB_COL = "publisher_clean"
DATE_COL = "date"

df_full = df_full.dropna(subset=[TEXT_COL]).reset_index(drop=True)
df_full[TEXT_COL] = df_full[TEXT_COL].astype(str)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
全量: 1189862 行
['title', 'summary', 'text', 'url', 'domain', 'publisher_clean', 'date', 'datetime', 'year', 'month', 'day', 'year_month_day', 'title_len', 'summary_len', 'text_len']


In [9]:
df = df_full.groupby(PUB_COL, group_keys=False).apply(
    lambda x: x.sample(frac=0.15, random_state=42)
).reset_index(drop=True)

print(f"采样后: {len(df)} 行")
print(df[PUB_COL].value_counts())

del df_full
import gc; gc.collect()

/tmp/ipykernel_11993/1862582093.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df_full.groupby(PUB_COL, group_keys=False).apply(


采样后: 178481 行
publisher_clean
nytimes           27788
washingtonpost    17436
foxnews           14326
theguardian       10605
nydailynews       10141
wsj                9128
usatoday           8147
cnn                7847
time               7746
mashable           5791
people             5612
forbes             5495
fortune            4206
cnbc               4011
foxsports          3997
telegraph          3814
9news.com          3690
tmz                3468
latimes            3336
bostonglobe        3111
thesun             2565
sfgate             2540
cbc                2432
aol                2271
nypost             1895
bbc                1800
abcnews            1748
nbcnews            1410
reuters            1082
bloomberg          1043
Name: count, dtype: int64


0

In [10]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

sia = SentimentIntensityAnalyzer()

print("Running VADER...")
vader_results = df[TEXT_COL].progress_apply(lambda x: sia.polarity_scores(x))
df['vader_compound'] = vader_results.apply(lambda x: x['compound'])
df['vader_neg']      = vader_results.apply(lambda x: x['neg'])
df['vader_pos']      = vader_results.apply(lambda x: x['pos'])
print("VADER done.")

Running VADER...


100%|██████████| 178481/178481 [1:05:29<00:00, 45.42it/s]


VADER done.


In [11]:
from afinn import Afinn

afinn = Afinn()

print("Running AFINN...")
df['afinn_score'] = df[TEXT_COL].progress_apply(afinn.score)
df['afinn_norm']  = df.apply(
    lambda r: r['afinn_score'] / max(len(r[TEXT_COL].split()), 1), axis=1
)
print("AFINN done.")

Running AFINN...


100%|██████████| 178481/178481 [1:01:32<00:00, 48.34it/s]


AFINN done.


In [19]:
from nrclex import NRCLex
import pprint

#e = NRCLex("I am happy and angry")
#pprint.pprint(e.__dict__)

TARGET_EMOTIONS = ['anger', 'fear', 'disgust']

def nrc_scores(text):
    try:
        words = text.lower().split()[:500]
        counts = {e: 0 for e in TARGET_EMOTIONS}
        for w in words:
            emotions = NRCLex.__lexicon__.get(w, [])
            for e in TARGET_EMOTIONS:
                if e in emotions:
                    counts[e] += 1
        total = max(len(words), 1)
        return {e: counts[e] / total for e in TARGET_EMOTIONS}
    except:
        return {e: 0.0 for e in TARGET_EMOTIONS}

print("Running NRC...")
nrc_results = df[TEXT_COL].progress_apply(nrc_scores)
nrc_df = pd.DataFrame(nrc_results.tolist())
df['nrc_anger']   = nrc_df['anger']
df['nrc_fear']    = nrc_df['fear']
df['nrc_disgust'] = nrc_df['disgust']
print("NRC done.")

Running NRC...


100%|██████████| 178481/178481 [00:25<00:00, 7007.80it/s]


NRC done.


In [20]:
df['IIS'] = (
    0.25 * (-df['vader_compound'])
  + 0.15 * df['vader_neg']
  + 0.20 * (-df['afinn_norm'])
  + 0.15 * df['nrc_anger']
  + 0.15 * df['nrc_fear']
  + 0.10 * df['nrc_disgust']
)

print(f"IIS统计:\n{df['IIS'].describe()}")

df['month'] = pd.to_datetime(df[DATE_COL], errors='coerce').dt.to_period('M')

summary = df.groupby([PUB_COL, 'month']).agg(
    avg_iis=('IIS', 'mean'),
    median_iis=('IIS', 'median'),
    max_iis=('IIS', 'max'),
    article_count=('IIS', 'count')
).reset_index()

print(f"聚合结果 ({len(summary)} rows):")
print(summary.head(10))

IIS统计:
count    178481.000000
mean         -0.074396
std           0.223518
min          -0.369975
25%          -0.248228
50%          -0.222978
75%           0.211719
max           0.545000
Name: IIS, dtype: float64
聚合结果 (30 rows):
  publisher_clean    month   avg_iis  median_iis   max_iis  article_count
0       9news.com  1970-01  0.084246    0.214610  0.370430           3690
1         abcnews  1970-01 -0.076690   -0.213602  0.343707           1748
2             aol  1970-01 -0.108342   -0.243521  0.354529           2271
3             bbc  1970-01 -0.141389   -0.242740  0.355647           1800
4       bloomberg  1970-01 -0.119289   -0.237170  0.303819           1043
5     bostonglobe  1970-01 -0.149690   -0.241142  0.314337           3111
6             cbc  1970-01 -0.055038   -0.214685  0.348170           2432
7            cnbc  1970-01 -0.108542   -0.215646  0.330461           4011
8             cnn  1970-01 -0.026378   -0.194871  0.545000           7847
9          forbes  1970-01 

In [21]:
from transformers import pipeline
from scipy.stats import spearmanr, pearsonr

sentiment_pipe = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    device=0
)

VAL_SIZE = 2000
val_sample = df.sample(n=min(VAL_SIZE, len(df)), random_state=42).copy()

transformer_results = []
for text in tqdm(val_sample[TEXT_COL].tolist(), desc="Transformer"):
    try:
        result = sentiment_pipe(text[:512])[0]
        transformer_results.append(result)
    except:
        transformer_results.append({'label': 'UNKNOWN', 'score': 0.0})

val_sample['tf_label'] = [r['label'] for r in transformer_results]
val_sample['tf_score'] = [r['score'] for r in transformer_results]
val_sample['tf_numeric'] = val_sample.apply(
    lambda r: r['tf_score'] if r['tf_label'] == 'NEGATIVE' else -r['tf_score'], axis=1
)

corr_pearson, p_pearson = pearsonr(val_sample['IIS'], val_sample['tf_numeric'])
corr_spearman, p_spearman = spearmanr(val_sample['IIS'], val_sample['tf_numeric'])

print(f"\n=== Validation Results (n={len(val_sample)}) ===")
print(f"Pearson r  = {corr_pearson:.4f}  (p={p_pearson:.2e})")
print(f"Spearman ρ = {corr_spearman:.4f} (p={p_spearman:.2e})")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

Transformer: 100%|██████████| 2000/2000 [00:11<00:00, 173.45it/s]



=== Validation Results (n=2000) ===
Pearson r  = 0.2875  (p=2.32e-39)
Spearman ρ = 0.3384 (p=8.82e-55)


In [22]:
df.to_csv(OUTPUT_DIR + "newsroom_with_iis_sampled.csv", index=False)
summary.to_csv(OUTPUT_DIR + "iis_summary_by_pub_month.csv", index=False)
val_sample.to_csv(OUTPUT_DIR + "iis_transformer_validation.csv", index=False)

print(f"文件已保存到 {OUTPUT_DIR}")
print("Done!")

文件已保存到 /content/drive/MyDrive/312Final/
Done!
